# Fine-tune, save, reload, predict

In Colab, select a GPU under **Runtime → Change runtime type** if available, then run all cells in order; the first code cell installs GRASP. Locally, run from the repository root after `pip install -e .`. This *illustrative* dataset uses RDKit-computed logP for 30 simple SMILES. It is not an experimental benchmark and does not reproduce paper scores. For real work, replace the CSVs with independent training and structure-separated validation data. Download the saved predictor directory before your Colab runtime ends if you want to keep it.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab
except ImportError:
    pass
else:
    repo = Path('/content/GRASP')
    if not repo.is_dir():
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/caithmac/GRASP.git', str(repo)], check=True)
    os.chdir(repo)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)


In [ ]:
from pathlib import Path
import torch
from scripts.make_demo_data import main as make_demo_data
from grasp.train import fit
from grasp import GRASPPredictor

make_demo_data()
model_source = 'hf_model' if Path('hf_model/model.safetensors').is_file() else 'caithmac/GRASP'


In [ ]:
result = fit(model_source=model_source,
             train_csv='examples/data/demo_train.csv',
             valid_csv='examples/data/demo_valid.csv',
             output_dir='runs/notebook_demo',
             task='regression', method='full',
             epochs=1, patience=1, batch_size=24,
             device='cuda' if torch.cuda.is_available() else 'cpu')
print(result)


In [ ]:
predictor = GRASPPredictor.from_pretrained('runs/notebook_demo')
new_smiles = ['CCO', 'CCOC', 'c1ccccc1']
print(list(zip(new_smiles, predictor.predict(new_smiles))))
